In [ ]:
using Bessels
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/simulator.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/visualization/plot_settings.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/visualization/plot_utils.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.9#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 3e-3*κ_2 #from Leon's paper

# Check for the Mean Field

In [ ]:
t_max=250/κ_b
re_a_0, im_a_0 = 0, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment)

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a[end-5000:end], q_CL_a[end-5000:end], t_list_CL[end-5000:end], ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

# Check for the chain

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/baths.jl")
using MPSDynamics

In [ ]:
n=7000

bath=Exp_decay
bath_params =(Ω=50*2*pi, ω_cut_off=20*2*pi)

AB, J = bath(bath_params=bath_params,κ=κ_b, C_inv=C_b_inv)
chain_params = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=7000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

In [ ]:
chain_params

In [ ]:
t_max=30/κ_b
ϕ_a_0, q_a_0=1.0, 0.0
ϕ_b_0, q_b_0=1.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModes(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
environment = OneChain(n, chain_params...)

ϕ_a,ϕ_b,q_a,q_b,t_list = simulator(t_max, system, drive, environment);

In [ ]:
plot(t_list*κ_b,ϕ_b)

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

F,freqs=FFT_perso(ϕ_b, t_list)
f_shifted=abs(freqs[argmax(abs.(F))])
plot(freqs,abs.(F))
vline!([f_shifted], linestyle=:dash, color=:red, label="f_shifted")
plot!(xlims=(4,8))

In [ ]:
Lamb_shift = f_shifted - ω_b_0/(2*pi)

In [ ]:
t_max=12/κ_b
ϕ_a_0, q_a_0=0.0, 0.0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = OneChain(n, chain_params...)

ϕ_a,ϕ_b,q_a,q_b,t_list = simulator(t_max, system, drive, environment)

In [ ]:
p1,p2 = plot_mode(ϕ_a, q_a, t_list, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
3*2*pi/((0.5*2*pi)^2-(3*2*pi)^2)

In [ ]:
3*2*pi/((3*0.5*2*pi)^2-(3*2*pi)^2)

In [ ]:
3*2*pi/((5*0.5*2*pi)^2-(3*2*pi)^2)

In [ ]:
3*2*pi/((7*0.5*2*pi)^2-(3*2*pi)^2)

In [ ]:
3*2*pi/((9*0.5*2*pi)^2-(3*2*pi)^2)

In [ ]:
3*2*pi/((11*0.5*2*pi)^2-(3*2*pi)^2)

# Parity cos-sin

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b = 2*ω_a

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b
ω_p=2ω_a-ω_d

ϵ_0 = 0.9#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 3e-3*κ_2 #from Leon's paper

In [ ]:
t_max=30/κ_b
re_a_0, im_a_0 = 1, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

In [ ]:
ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment, equation = equation_parity_cos_sin!)

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a[end-5000:end], q_CL_a[end-5000:end], t_list_CL[end-5000:end], ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
p1,p2 = plot_mode(ϕ_CL_b[end-5000:end], q_CL_b[end-5000:end], t_list_CL[end-5000:end], ω_b, Z_b)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(b)", ylabel=L"\Im(b)",
title=["Resonator b" "Resonator b RF"],
legend=false)